In [4]:
"""
Titanic Data Analysis and JSON Export
Author: [Bhabishwor]
Description: Analyze Titanic passenger data, engineer features, and export to JSON
"""
 
import pandas as pd
import numpy as np
import json
from pathlib import Path
 
# Set up paths
DATA_DIR = Path("data")
CSV_FILE = DATA_DIR / "titanic.csv"
JSON_FILE = DATA_DIR / "titanic_data.json"
 
# Create data directory if it doesn't exist
DATA_DIR.mkdir(exist_ok=True)
 
print("Project setup complete!")
print(f"Data directory: {DATA_DIR}")
print(f"CSV file location: {CSV_FILE}")

Project setup complete!
Data directory: data
CSV file location: data\titanic.csv


In [5]:
df = pd.read_csv(CSV_FILE)
print(f"Dataset loaded successfully! Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst few rows:")
print(df.head())

Dataset loaded successfully! Shape: (891, 12)

Columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']

First few rows:
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 175

In [6]:
#step 3: calculating descriptive statistics for numerical columns

print("\n" + "=" * 50)
print("DESCRIPTIVE STATISTICS")
print("=" * 50)

# Select numeric columns
numeric_columns = df.select_dtypes(include="number")

print("\nNumeric columns:")
print(list(numeric_columns.columns))

# Calculate statistics
means = numeric_columns.mean()
medians = numeric_columns.median()
standard_deviations = numeric_columns.std()

print("\nMeans:")
print(means)

print("\nMedians:")
print(medians)

print("\nStandard deviations:")
print(standard_deviations)


DESCRIPTIVE STATISTICS

Numeric columns:
['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']

Means:
PassengerId    446.000000
Survived         0.383838
Pclass           2.308642
Age             29.699118
SibSp            0.523008
Parch            0.381594
Fare            32.204208
dtype: float64

Medians:
PassengerId    446.0000
Survived         0.0000
Pclass           3.0000
Age             28.0000
SibSp            0.0000
Parch            0.0000
Fare            14.4542
dtype: float64

Standard deviations:
PassengerId    257.353842
Survived         0.486592
Pclass           0.836071
Age             14.526497
SibSp            1.102743
Parch            0.806057
Fare            49.693429
dtype: float64


In [7]:
# Step 4: Identifying missing values

print("\n" + "=" * 50)
print("MISSING VALUES ANALYSIS")
print("=" * 50)

missing_data = {}

for col in df.columns:
    missing_count = df[col].isna().sum()
    missing_percent = (missing_count / len(df)) * 100

    missing_data[col] = {
        "count": missing_count,
        "percentage": missing_percent
    }

    print(f"{col}: {missing_count} missing values "
          f"({missing_percent:.2f}%)")


MISSING VALUES ANALYSIS
PassengerId: 0 missing values (0.00%)
Survived: 0 missing values (0.00%)
Pclass: 0 missing values (0.00%)
Name: 0 missing values (0.00%)
Sex: 0 missing values (0.00%)
Age: 177 missing values (19.87%)
SibSp: 0 missing values (0.00%)
Parch: 0 missing values (0.00%)
Ticket: 0 missing values (0.00%)
Fare: 0 missing values (0.00%)
Cabin: 687 missing values (77.10%)
Embarked: 2 missing values (0.22%)


In [8]:
# Step 5: Feature Engineering

print("\n" + "=" * 50)
print("FEATURE ENGINEERING")
print("=" * 50)

# Work on a copy so the original DataFrame remains unchanged
df_features = df.copy()

# Feature 1: FamilySize
df_features["FamilySize"] = (
    df_features["SibSp"] + df_features["Parch"] + 1
)

print("\nFamilySize:")
print(df_features[["SibSp", "Parch", "FamilySize"]].head(10))


FEATURE ENGINEERING

FamilySize:
   SibSp  Parch  FamilySize
0      1      0           2
1      1      0           2
2      0      0           1
3      1      0           2
4      0      0           1
5      0      0           1
6      0      0           1
7      3      1           5
8      0      2           3
9      1      0           2


In [9]:
# Feature 2: IsAlone
df_features["IsAlone"] = df_features["FamilySize"] == 1

print("\nIsAlone:")
print(df_features[["FamilySize", "IsAlone"]].head(10))


IsAlone:
   FamilySize  IsAlone
0           2    False
1           2    False
2           1     True
3           2    False
4           1     True
5           1     True
6           1     True
7           5    False
8           3    False
9           2    False


In [10]:
# Feature 3: AgeGroup

def categorize_age(age):
    """Place an age into an age category."""
    if pd.isna(age):
        return "Unknown"
    elif age < 18:
        return "Child"
    elif age < 30:
        return "Young Adult"
    elif age < 50:
        return "Adult"
    else:
        return "Senior"


df_features["AgeGroup"] = df_features["Age"].apply(categorize_age)

print("\nAgeGroup:")
print(df_features[["Age", "AgeGroup"]].head(10))


AgeGroup:
    Age     AgeGroup
0  22.0  Young Adult
1  38.0        Adult
2  26.0  Young Adult
3  35.0        Adult
4  35.0        Adult
5   NaN      Unknown
6  54.0       Senior
7   2.0        Child
8  27.0  Young Adult
9  14.0        Child


In [11]:
#Analyze feature differences between survivors and non-survivors
#but first separate the data into two groups based on survival status

#  survived = df_features[df_features["Survived"] == 1]
not_survived = df_features[df_features["Survived"] == 0]

In [12]:
#then comparing family size between survivors and non-survivors
print("\nFamily size by survival status:")

family_survival = df_features.groupby("Survived")["FamilySize"].agg(
    ["mean", "median", "std"]
)

print(family_survival)


Family size by survival status:
              mean  median       std
Survived                            
0         1.883424     1.0  1.830669
1         1.938596     2.0  1.186076


In [13]:
# Separate the two survival groups
survived = df_features[df_features["Survived"] == 1]
not_survived = df_features[df_features["Survived"] == 0]

In [14]:
#comparing the average family size directly
print(f"\nSurvived family-size mean: "
      f"{survived['FamilySize'].mean():.2f}")

print(f"Did not survive family-size mean: "
      f"{not_survived['FamilySize'].mean():.2f}")


Survived family-size mean: 1.94
Did not survive family-size mean: 1.88


In [15]:
#Finally, examine the age groups
print("\nSurvival rate by age group:")

age_group_survival = df_features.groupby("AgeGroup")["Survived"].mean()
print(age_group_survival)


Survival rate by age group:
AgeGroup
Adult          0.417969
Child          0.539823
Senior         0.364865
Unknown        0.293785
Young Adult    0.350554
Name: Survived, dtype: float64


In [16]:
print("\nSurvival counts by age group:")
print(pd.crosstab(df_features["AgeGroup"], df_features["Survived"]))


Survival counts by age group:
Survived       0    1
AgeGroup             
Adult        149  107
Child         52   61
Senior        47   27
Unknown      125   52
Young Adult  176   95


In [17]:
print(df_features[["FamilySize", "IsAlone", "AgeGroup"]].head())

   FamilySize  IsAlone     AgeGroup
0           2    False  Young Adult
1           2    False        Adult
2           1     True  Young Adult
3           2    False        Adult
4           1     True        Adult


In [18]:
# Step 6: Classes for JSON Export

class Passenger:
    """Represents one Titanic passenger."""

    def __init__(
        self,
        passenger_id,
        name,
        age,
        sex,
        survived,
        pclass,
        fare,
        embarked=None,
        family_size=None,
        is_alone=None,
        title=None
    ):
        # Convert missing values to None
        self.passenger_id = (
            int(passenger_id) if pd.notna(passenger_id) else None
        )

        self.name = name if pd.notna(name) else None
        self.age = float(age) if pd.notna(age) else None
        self.sex = sex if pd.notna(sex) else None
        self.survived = int(survived) if pd.notna(survived) else None
        self.pclass = int(pclass) if pd.notna(pclass) else None
        self.fare = float(fare) if pd.notna(fare) else None
        self.embarked = embarked if pd.notna(embarked) else None

        self.family_size = (
            int(family_size) if pd.notna(family_size) else None
        )

        self.is_alone = (
            bool(is_alone) if pd.notna(is_alone) else None
        )

        self.title = title if pd.notna(title) else None

    def to_dict(self):
        """Convert this passenger into a JSON-compatible dictionary."""
        return {
            "passenger_id": self.passenger_id,
            "name": self.name,
            "age": self.age,
            "sex": self.sex,
            "survived": self.survived,
            "pclass": self.pclass,
            "fare": self.fare,
            "embarked": self.embarked,
            "family_size": self.family_size,
            "is_alone": self.is_alone,
            "title": self.title
        }

In [19]:
#Testing the class with a sample passenger just one row from the DataFrame
first_row = df_features.iloc[0]

passenger = Passenger(
    passenger_id=first_row["PassengerId"],
    name=first_row["Name"],
    age=first_row["Age"],
    sex=first_row["Sex"],
    survived=first_row["Survived"],
    pclass=first_row["Pclass"],
    fare=first_row["Fare"],
    embarked=first_row["Embarked"],
    family_size=first_row["FamilySize"],
    is_alone=first_row["IsAlone"]
)

print(passenger.to_dict())

{'passenger_id': 1, 'name': 'Braund, Mr. Owen Harris', 'age': 22.0, 'sex': 'male', 'survived': 0, 'pclass': 3, 'fare': 7.25, 'embarked': 'S', 'family_size': 2, 'is_alone': False, 'title': None}


In [20]:
first_row = df_features.iloc[0]

test_passenger = Passenger(
    passenger_id=first_row["PassengerId"],
    name=first_row["Name"],
    age=first_row["Age"],
    sex=first_row["Sex"],
    survived=first_row["Survived"],
    pclass=first_row["Pclass"],
    fare=first_row["Fare"],
    embarked=first_row["Embarked"],
    family_size=first_row["FamilySize"],
    is_alone=first_row["IsAlone"]
)

print(test_passenger.to_dict())

{'passenger_id': 1, 'name': 'Braund, Mr. Owen Harris', 'age': 22.0, 'sex': 'male', 'survived': 0, 'pclass': 3, 'fare': 7.25, 'embarked': 'S', 'family_size': 2, 'is_alone': False, 'title': None}


In [21]:
class TitanicDataset:
    """Manages the processed Titanic dataset."""

    def __init__(self, dataframe):
        self.dataframe = dataframe
        self.passengers = []

    def create_passengers(self):
        """Create Passenger objects from dataframe rows."""
        # TODO: loop through self.dataframe.iterrows()
        # TODO: create one Passenger object per row
        # TODO: append each object to self.passengers
        pass

    def to_dict(self):
        """Return the dataset in a JSON-compatible structure."""
        # TODO: convert each Passenger object using passenger.to_dict()
        pass

    def export_to_json(self, file_path):
        """Export the dataset to a JSON file."""
        # TODO: call create_passengers()
        # TODO: open file_path for writing
        # TODO: use json.dump(...)
        pass

In [22]:
dataset = TitanicDataset(df_features)

In [23]:
print([name for name in dir(dataset) if not name.startswith("_")])

['create_passengers', 'dataframe', 'export_to_json', 'passengers', 'to_dict']


In [24]:
# Step 7: Check which parts currently exist

required_names = [
    "df",
    "df_features",
    "Passenger",
    "TitanicDataset",
    "JSON_FILE"
]

for name in required_names:
    if name in globals():
        print(f"✓ {name} is available")
    else:
        print(f"✗ {name} is missing")

✓ df is available
✓ df_features is available
✓ Passenger is available
✓ TitanicDataset is available
✓ JSON_FILE is available


In [25]:
# Validate data loading
print("Dataset shape:", df.shape)
assert not df.empty

# Validate engineered features
for feature in ["FamilySize", "IsAlone", "AgeGroup"]:
    assert feature in df_features.columns
    print(f"✓ {feature} exists")

# Validate missing-value analysis
missing_counts = df.isna().sum()
print("\nMissing values:")
print(missing_counts)

# Validate Passenger class
first_row = df_features.iloc[0]

test_passenger = Passenger(
    passenger_id=first_row["PassengerId"],
    name=first_row["Name"],
    age=first_row["Age"],
    sex=first_row["Sex"],
    survived=first_row["Survived"],
    pclass=first_row["Pclass"],
    fare=first_row["Fare"],
    embarked=first_row["Embarked"],
    family_size=first_row["FamilySize"],
    is_alone=first_row["IsAlone"]
)

print("\nPassenger dictionary:")
print(test_passenger.to_dict())

Dataset shape: (891, 12)
✓ FamilySize exists
✓ IsAlone exists
✓ AgeGroup exists

Missing values:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

Passenger dictionary:
{'passenger_id': 1, 'name': 'Braund, Mr. Owen Harris', 'age': 22.0, 'sex': 'male', 'survived': 0, 'pclass': 3, 'fare': 7.25, 'embarked': 'S', 'family_size': 2, 'is_alone': False, 'title': None}


In [26]:
print(JSON_FILE.resolve())
print(JSON_FILE.exists())

C:\Users\deuts\Documents\AI-AA\bootcamp-practice\L2.1 Titanic\data\titanic_data.json
True


In [27]:
passenger_records = []

for _, row in df_features.iterrows():
    passenger = Passenger(
        passenger_id=row["PassengerId"],
        name=row["Name"],
        age=row["Age"],
        sex=row["Sex"],
        survived=row["Survived"],
        pclass=row["Pclass"],
        fare=row["Fare"],
        embarked=row["Embarked"],
        family_size=row["FamilySize"],
        is_alone=row["IsAlone"]
    )

    passenger_records.append(passenger.to_dict())

In [28]:
DATA_DIR.mkdir(exist_ok=True)

with open(JSON_FILE, "w", encoding="utf-8") as f:
    json.dump(passenger_records, f, indent=2)

print(f"JSON file created at: {JSON_FILE.resolve()}")

JSON file created at: C:\Users\deuts\Documents\AI-AA\bootcamp-practice\L2.1 Titanic\data\titanic_data.json


In [29]:
print(JSON_FILE.exists())

True


In [30]:
with open(JSON_FILE, "r", encoding="utf-8") as f:
    json_data = json.load(f)

print("JSON validation passed")
print(f"Records loaded: {len(json_data)}")

JSON validation passed
Records loaded: 891


In [31]:
JSON_FILE = DATA_DIR / "titanic_data.json"

In [32]:
# Step 7: Testing and Validation

with open(JSON_FILE, "r", encoding="utf-8") as f:
    json_data = json.load(f)

print("JSON validation passed")
print(f"Number of records: {len(json_data)}")
print(f"Data type: {type(json_data).__name__}")

JSON validation passed
Number of records: 891
Data type: list


In [33]:
# Confirm that JSON contains records
assert isinstance(json_data, list)
assert len(json_data) == len(df_features)

# Confirm that the first record is a dictionary
assert isinstance(json_data[0], dict)

# Confirm expected fields exist
expected_fields = [
    "passenger_id",
    "name",
    "age",
    "sex",
    "survived",
    "pclass",
    "fare",
    "embarked",
    "family_size",
    "is_alone",
    "title"
]

for field in expected_fields:
    assert field in json_data[0], f"Missing field: {field}"

print("JSON structure validation passed")
print("\nFirst passenger record:")
print(json_data[0])

JSON structure validation passed

First passenger record:
{'passenger_id': 1, 'name': 'Braund, Mr. Owen Harris', 'age': 22.0, 'sex': 'male', 'survived': 0, 'pclass': 3, 'fare': 7.25, 'embarked': 'S', 'family_size': 2, 'is_alone': False, 'title': None}
